# GD + Thaler для FCN4/MNIST

Ноутбук содержит модификационный эксперимент для ВКР: полный градиентный спуск дополняется детерминированным возмущением, построенным по орбите отображения Талера.

В ноутбуке выполняются:
- загрузка MNIST и построение модели FCN4;
- обучение в режиме GD + Thaler;
- оценка хвостового показателя весовых коэффициентов;
- разреживание обученной модели;
- сохранение таблиц, графиков и контрольной точки модели.

Запускать ячейки нужно последовательно сверху вниз.


In [ ]:
# 0. Импорт библиотек, фиксация seed, устройство и папки
# Ячейка готовит среду выполнения и создаёт каталоги для результатов.

import os
import math
import copy
import json
import random
import time
import zipfile
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

RESULTS_DIR = "results_gd_thaler"
FIGURES_DIR = "figures_gd_thaler"
CHECKPOINTS_DIR = "checkpoints_gd_thaler"

for d in [RESULTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 1. Основные настройки эксперимента
# Здесь задаются ширина сети, число шагов обучения, ранняя остановка и доли разреживания.

# Основной режим рассчитан на запуск в Google Colab.
WIDTH = 2048
TRAIN_SUBSET_SIZE = None
TEST_SUBSET_SIZE = 10000

# Размер технического батча для накопления полного градиента.
FULL_GRAD_BATCH_SIZE = 512

# Параметры ранней остановки и сохранения лучшей модели.
MAX_GD_STEPS = 200
LOG_EVERY = 10
TARGET_TEST_ACC = 0.94       # для малого/среднего Colab-запуска; для финала можно 0.95-0.97
PATIENCE_EVALS = 5           # сколько eval-точек ждать после достижения TARGET_TEST_ACC
MIN_STEPS_BEFORE_STOP = 80
USE_BEST_CHECKPOINT = True

RUN_THALER_SWEEP = True
RUN_PRUNING = True
RUN_SVD_PRUNING = True       # если SVD долго, поставить False

# keep_ratio — доля сохранённых параметров после разреживания.
PRUNING_KEEP_RATIOS = [1.0, 0.8, 0.6, 0.4, 0.2, 0.1]

print(dict(
    WIDTH=WIDTH,
    TRAIN_SUBSET_SIZE=TRAIN_SUBSET_SIZE,
    FULL_GRAD_BATCH_SIZE=FULL_GRAD_BATCH_SIZE,
    MAX_GD_STEPS=MAX_GD_STEPS,
    TARGET_TEST_ACC=TARGET_TEST_ACC,
    PATIENCE_EVALS=PATIENCE_EVALS,
    MIN_STEPS_BEFORE_STOP=MIN_STEPS_BEFORE_STOP,
    USE_BEST_CHECKPOINT=USE_BEST_CHECKPOINT,
    PRUNING_KEEP_RATIOS=PRUNING_KEEP_RATIOS,
))

In [ ]:
# 2. Данные MNIST и модель FCN4
# Ячейка загружает данные, строит полносвязную сеть и задаёт функцию проверки качества.

def get_dataloaders(
    dataset="MNIST",
    train_batch_size=512,
    test_batch_size=512,
    shuffle_train=False,
    train_subset_size=None,
    test_subset_size=None,
    seed=42,
):
    dataset = dataset.upper()
    if dataset != "MNIST":
        raise ValueError("В этом ноутбуке настроен эксперимент только для MNIST.")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_set = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_set = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    if train_subset_size is not None and train_subset_size < len(train_set):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(train_set), size=int(train_subset_size), replace=False)
        train_set = Subset(train_set, idx.tolist())

    if test_subset_size is not None and test_subset_size < len(test_set):
        rng = np.random.default_rng(seed + 1)
        idx = rng.choice(len(test_set), size=int(test_subset_size), replace=False)
        test_set = Subset(test_set, idx.tolist())

    train_loader = DataLoader(
        train_set,
        batch_size=train_batch_size,
        shuffle=shuffle_train,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_set,
        batch_size=test_batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, test_loader, 28 * 28, 10


class FCN4(nn.Module):
    """FCN4: 4 hidden fully-connected ReLU layers + output layer."""
    def __init__(self, input_dim=784, width=512, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, num_classes),
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


def build_fcn4(width=512):
    return FCN4(input_dim=784, width=width, num_classes=10)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")
        total_loss += loss.item()
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    return total_loss / max(total, 1), correct / max(total, 1)


train_loader, test_loader, _, _ = get_dataloaders(
    train_batch_size=FULL_GRAD_BATCH_SIZE,
    test_batch_size=512,
    shuffle_train=False,
    train_subset_size=TRAIN_SUBSET_SIZE,
    test_subset_size=TEST_SUBSET_SIZE,
    seed=SEED,
)

_tmp_model = build_fcn4(width=WIDTH)
print("Train size:", len(train_loader.dataset))
print("Test size:", len(test_loader.dataset))
print("FCN4 parameters:", count_parameters(_tmp_model))
del _tmp_model

In [ ]:
# 3. Оценка хвостового показателя
# Реализуется многомерный MMO-оцениватель, применяемый к весовым слоям модели.

def find_m_barsbey(N, m=0):
    if m == 0:
        m = int(np.sqrt(N))
    m = max(2, int(m))
    while m > 1 and N % m != 0:
        m -= 1
    return m


def alpha_estimator_multi_barsbey(m, X):
    """
    Multivariate MMO estimator.
    X: N x d matrix. K=N, K1=m, K2=N/m.
    1/alpha = (mean log ||sum within groups|| - mean log ||X_i||) / log(m).
    """
    if not torch.is_tensor(X):
        X = torch.from_numpy(np.asarray(X, dtype=np.float64))
    X = X.to(dtype=torch.float64, device="cpu")
    N = X.shape[0]
    if m <= 1 or N < 2 * m or N % m != 0:
        return np.nan
    n = int(N / m)
    try:
        Y = torch.sum(X.view(n, m, -1), dim=1)
    except RuntimeError:
        return np.nan
    eps = np.spacing(1)
    Y_log_norm = torch.log(torch.linalg.norm(Y, dim=1) + eps).mean()
    X_log_norm = torch.log(torch.linalg.norm(X, dim=1) + eps).mean()
    diff = (Y_log_norm - X_log_norm) / math.log(m)
    if (not torch.isfinite(diff)) or diff.item() <= 0:
        return np.nan
    alpha = 1.0 / diff.item()
    if not np.isfinite(alpha):
        return np.nan
    return float(alpha)


def est_alpha_barsbey_matrix(X, clip_to_2=True):
    """
    Estimate alpha for one weight matrix.
    The matrix is centered by median before estimation, as in the article/pruning setup.
    """
    if torch.is_tensor(X):
        X = X.detach().cpu().numpy()
    X = np.asarray(X, dtype=np.float64)

    if X.ndim == 1:
        X = X.reshape(-1, 1)
    elif X.ndim > 2:
        X = X.reshape(X.shape[0], -1)

    if X.shape[0] < 20:
        return np.nan

    med = np.median(X, axis=0, keepdims=True)
    X = X - med

    N = X.shape[0]
    candidate_ms = [2, 5, 10, 20, 50, 100, 500, 1000, int(np.sqrt(N))]
    alphas = []
    for m0 in candidate_ms:
        m = find_m_barsbey(N, m0)
        if m <= 1 or N < 2 * m or N % m != 0:
            continue
        a = alpha_estimator_multi_barsbey(m, X)
        if np.isfinite(a):
            if clip_to_2:
                a = min(a, 2.0)
            alphas.append(a)

    if len(alphas) == 0:
        return np.nan
    return float(np.median(alphas))


def estimate_model_alphas(model, skip_last=True):
    rows = []
    weight_items = [(n, p) for n, p in model.named_parameters() if "weight" in n and p.ndim >= 2]
    if skip_last and len(weight_items) > 1:
        weight_items_for_mean = weight_items[:-1]
    else:
        weight_items_for_mean = weight_items

    mean_values = []
    for layer_id, (name, p) in enumerate(weight_items):
        W = p.detach().cpu().numpy()
        W2 = W.reshape(W.shape[0], -1)
        alpha = est_alpha_barsbey_matrix(W2, clip_to_2=True)
        use_for_mean = (name, p) in weight_items_for_mean
        if use_for_mean and np.isfinite(alpha):
            mean_values.append(alpha)
        rows.append({
            "layer_id": layer_id,
            "name": name,
            "shape": tuple(W.shape),
            "alpha": alpha,
            "used_for_mean": use_for_mean,
        })

    mean_alpha = float(np.mean(mean_values)) if len(mean_values) else np.nan
    return mean_alpha, pd.DataFrame(rows)

## Детерминированный шум Талера

Сигнал строится по орбите отображения Талера и затем добавляется к полному градиентному шагу.  
Параметры `s`, `target_alpha_noise`, `eps`, `inner_steps` и `noise_scale` задаются в конфигурации эксперимента.

Основной режим шума — `layerwise`: для каждого обучаемого тензора используется свой детерминированный генератор. Это сохраняет воспроизводимость и распределяет возмущение по весовым коэффициентам сети.


In [ ]:
# 4. Генератор детерминированного шума Талера
# Класс строит тяжёлохвостный сигнал по орбите отображения Талера.

class ThalerNoiseGenerator:
    """
    Deterministic heavy-tailed signal from T_s(x) = (x + x^(1+s)) mod 1.
    Observable: phi(x) = sign(sin(2*pi*x)) * x^(-beta) * 1{x < eps}.
    """
    def __init__(
        self,
        s=0.5,
        beta=0.3,
        eps=0.05,
        x0=0.123456789,
        burn_in=1000,
        inner_steps=1,
        aggregate="sum_sqrt",
        clip_raw=1e6,
        calibrate_size=8000,
    ):
        self.s = float(s)
        self.beta = float(beta)
        self.eps = float(eps)
        self.x = float(x0 % 1.0)
        self.x0 = float(x0 % 1.0)
        self.burn_in = int(burn_in)
        self.inner_steps = int(inner_steps)
        self.aggregate = aggregate
        self.clip_raw = float(clip_raw)
        self.calibrate_size = int(calibrate_size)
        self.n_iter = 0

        for _ in range(self.burn_in):
            self._step_map()

        calib = np.array([self._raw_next_aggregate() for _ in range(self.calibrate_size)], dtype=np.float64)
        finite = np.isfinite(calib)
        calib = calib[finite]
        if len(calib) == 0:
            self.center = 0.0
            self.scale = 1.0
        else:
            self.center = float(np.median(calib))
            mad = float(np.median(np.abs(calib - self.center)))
            if (not np.isfinite(mad)) or mad < 1e-12:
                std = float(np.std(calib))
                self.scale = std if np.isfinite(std) and std > 1e-12 else 1.0
            else:
                self.scale = mad / 0.6744897501960817

        self.calib_mean = float(np.mean(calib)) if len(calib) else np.nan
        self.calib_std = float(np.std(calib)) if len(calib) else np.nan
        self.calib_min = float(np.min(calib)) if len(calib) else np.nan
        self.calib_max = float(np.max(calib)) if len(calib) else np.nan

    def _step_map(self):
        # Avoid exact 0 under finite precision.
        if self.x <= 0.0:
            self.x = np.nextafter(0.0, 1.0)
        self.x = (self.x + self.x ** (1.0 + self.s)) % 1.0
        self.n_iter += 1
        if self.x <= 0.0:
            self.x = np.nextafter(0.0, 1.0)
        return self.x

    def _observable(self, x):
        if x < self.eps:
            val = math.sin(2.0 * math.pi * x)
            sign = 1.0 if val >= 0.0 else -1.0
            out = sign * (max(x, np.nextafter(0.0, 1.0)) ** (-self.beta))
            out = float(np.clip(out, -self.clip_raw, self.clip_raw))
            return out
        return 0.0

    def _raw_next_single(self):
        x = self._step_map()
        return self._observable(x)

    def _raw_next_aggregate(self):
        vals = [self._raw_next_single() for _ in range(self.inner_steps)]
        vals = np.asarray(vals, dtype=np.float64)
        if self.aggregate == "sum_sqrt":
            return float(vals.sum() / math.sqrt(max(self.inner_steps, 1)))
        if self.aggregate == "mean":
            return float(vals.mean())
        if self.aggregate == "maxabs":
            j = int(np.argmax(np.abs(vals)))
            return float(vals[j])
        if self.aggregate == "sum":
            return float(vals.sum())
        raise ValueError(f"Unknown aggregate={self.aggregate}")

    def next(self):
        raw = self._raw_next_aggregate()
        z = (raw - self.center) / self.scale
        if not np.isfinite(z):
            z = 0.0
        return float(z), float(raw), float(self.x)

    def sample(self, n):
        rows = []
        for _ in range(int(n)):
            z, raw, x = self.next()
            rows.append((z, raw, x))
        return np.array(rows, dtype=np.float64)


def hill_tail_index_abs(values, top_frac=0.05):
    """Simple diagnostic only for the noise sequence; weight alpha uses Barsbey MMO above."""
    x = np.abs(np.asarray(values, dtype=np.float64))
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) < 50:
        return np.nan
    x = np.sort(x)[::-1]
    k = max(10, int(len(x) * top_frac))
    k = min(k, len(x) - 1)
    threshold = x[k]
    if threshold <= 0:
        return np.nan
    gamma = np.mean(np.log(x[:k]) - np.log(threshold))
    if gamma <= 0 or not np.isfinite(gamma):
        return np.nan
    return float(1.0 / gamma)


def make_deterministic_directions(model, seed=12345):
    """
    Fixed deterministic Rademacher direction for each trainable tensor.
    Each tensor direction has norm 1, so noise_scale is comparable across tensors.
    """
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))
    directions = {}
    for name, p in model.named_parameters():
        if p.requires_grad:
            r = torch.randint(0, 2, p.shape, generator=gen, dtype=torch.int8)
            v = (2 * r.float() - 1.0).to(device)
            v = v / math.sqrt(v.numel())
            directions[name] = v
    return directions


def shifted_x0(base_x0, j):
    """
    Deterministic shift of x0 for layerwise generators.
    Keeps values away from exact 0 and 1.
    """
    y = (float(base_x0) + (j + 1) * 0.137137137) % 1.0
    if y <= 1e-12:
        y += 0.123456789
    return float(y % 1.0)


def make_thaler_generators_for_model(model, config):
    """
    noise_mode:
      - "global": one deterministic signal z_k for all tensors;
      - "layerwise": independent deterministic Thaler signal per trainable tensor.
    """
    if not config.use_thaler:
        return {}

    if config.noise_mode == "global":
        return {
            "__global__": ThalerNoiseGenerator(
                s=config.s, beta=config.beta, eps=config.eps, x0=config.x0,
                burn_in=config.burn_in, inner_steps=config.inner_steps,
                aggregate=config.aggregate, clip_raw=config.clip_raw,
                calibrate_size=config.calibrate_size,
            )
        }

    if config.noise_mode == "layerwise":
        generators = {}
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        for j, name in enumerate(trainable_names):
            generators[name] = ThalerNoiseGenerator(
                s=config.s, beta=config.beta, eps=config.eps, x0=shifted_x0(config.x0, j),
                burn_in=config.burn_in, inner_steps=config.inner_steps,
                aggregate=config.aggregate, clip_raw=config.clip_raw,
                calibrate_size=config.calibrate_size,
            )
        return generators

    raise ValueError(f"Unknown noise_mode={config.noise_mode}")


def next_noise_values(generators, model):
    """
    Returns:
      z_by_name: dict name -> normalized z
      raw_by_name: dict name -> raw observable
      x_by_name: dict name -> current orbit state
    """
    if not generators:
        return {}, {}, {}

    if "__global__" in generators:
        z, raw, x = generators["__global__"].next()
        z_by_name = {}
        raw_by_name = {}
        x_by_name = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                z_by_name[name] = z
                raw_by_name[name] = raw
                x_by_name[name] = x
        return z_by_name, raw_by_name, x_by_name

    z_by_name = {}
    raw_by_name = {}
    x_by_name = {}
    for name, gen in generators.items():
        z, raw, x = gen.next()
        z_by_name[name] = z
        raw_by_name[name] = raw
        x_by_name[name] = x
    return z_by_name, raw_by_name, x_by_name


def add_thaler_noise(model, directions, z_by_name, noise_scale):
    """
    Adds sigma * z_name * v_name to each trainable tensor.
    Returns Euclidean norm of the actually added noise vector.
    """
    if not z_by_name or float(noise_scale) == 0.0:
        return 0.0

    noise_sq = 0.0
    with torch.no_grad():
        for name, p in model.named_parameters():
            if p.requires_grad and name in directions:
                coeff = float(noise_scale) * float(z_by_name.get(name, 0.0))
                if coeff != 0.0:
                    p.add_(directions[name], alpha=coeff)
                    # directions[name] has norm 1 per tensor
                    noise_sq += coeff * coeff * float(torch.sum(directions[name] ** 2).detach().cpu())
    return float(math.sqrt(max(noise_sq, 0.0)))


def diagnose_thaler_signal(config, n=5000):
    beta = (1.0 - config["s"]) / config["target_alpha_noise"]
    gen = ThalerNoiseGenerator(
        s=config["s"], beta=beta, eps=config["eps"], x0=config["x0"],
        burn_in=config.get("burn_in", 1000), inner_steps=config["inner_steps"],
        aggregate=config.get("aggregate", "sum_sqrt"),
        clip_raw=config.get("clip_raw", 1e6),
        calibrate_size=config.get("calibrate_size", 8000),
    )
    arr = gen.sample(n)
    z = arr[:, 0]
    raw = arr[:, 1]
    x = arr[:, 2]
    diag = {
        "s": config["s"],
        "target_alpha_noise": config["target_alpha_noise"],
        "beta": beta,
        "eps": config["eps"],
        "x0": config["x0"],
        "inner_steps": config["inner_steps"],
        "aggregate": config.get("aggregate", "sum_sqrt"),
        "z_min": float(np.min(z)),
        "z_max": float(np.max(z)),
        "z_mean": float(np.mean(z)),
        "z_std": float(np.std(z)),
        "z_nonzero_share": float(np.mean(np.abs(raw) > 0)),
        "z_unique_ratio": float(len(np.unique(np.round(z, 12))) / len(z)),
        "hill_alpha_abs_z": hill_tail_index_abs(z),
        "raw_min": float(np.min(raw)),
        "raw_max": float(np.max(raw)),
        "raw_mean": float(np.mean(raw)),
        "raw_std": float(np.std(raw)),
        "x_min": float(np.min(x)),
        "x_max": float(np.max(x)),
        "calib_center": gen.center,
        "calib_scale": gen.scale,
        "calib_raw_std": gen.calib_std,
    }
    return diag, arr

In [ ]:
# 5. Обучение GD + Thaler
# Полный градиент считается накоплением по обучающей выборке, затем добавляется детерминированное возмущение.

@dataclass
class GDThalerConfig:
    dataset: str = "MNIST"
    model_name: str = "FCN4"
    width: int = WIDTH
    lr: float = 0.03
    noise_scale: float = 0.002
    seed: int = 42
    direction_seed: int = 2026

    use_thaler: bool = True
    noise_mode: str = "layerwise"   # "layerwise" is more useful for transferring tails to weights

    s: float = 0.7
    target_alpha_noise: float = 1.4
    eps: float = 0.10
    x0: float = 0.123456789
    burn_in: int = 1000
    inner_steps: int = 20
    aggregate: str = "sum_sqrt"
    clip_raw: float = 1e6
    calibrate_size: int = 8000

    train_subset_size: int | None = TRAIN_SUBSET_SIZE
    full_grad_batch_size: int = FULL_GRAD_BATCH_SIZE
    max_gd_steps: int = MAX_GD_STEPS
    log_every: int = LOG_EVERY

    target_test_acc: float = TARGET_TEST_ACC
    patience_evals: int = PATIENCE_EVALS
    min_steps_before_stop: int = MIN_STEPS_BEFORE_STOP
    use_best_checkpoint: bool = USE_BEST_CHECKPOINT

    @property
    def beta(self):
        if self.target_alpha_noise <= 0:
            return 0.0
        return (1.0 - self.s) / self.target_alpha_noise


def full_gradient_step(model, train_loader, lr):
    """
    Exact GD step over train_loader.dataset, implemented by gradient accumulation.
    Returns train loss/acc and norms of the full gradient and GD update.
    """
    model.train()
    for p in model.parameters():
        if p.grad is not None:
            p.grad = None

    n_total = len(train_loader.dataset)
    total_loss = 0.0
    total = 0
    correct = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss_sum = F.cross_entropy(logits, y, reduction="sum")
        (loss_sum / n_total).backward()

        total_loss += loss_sum.item()
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    grad_sq = 0.0
    with torch.no_grad():
        for p in model.parameters():
            if p.grad is not None:
                grad_sq += float(torch.sum(p.grad.detach() ** 2).cpu())
        grad_norm = math.sqrt(max(grad_sq, 0.0))
        gd_update_norm = float(lr) * grad_norm

        for p in model.parameters():
            if p.grad is not None:
                p.add_(p.grad, alpha=-lr)

    return total_loss / max(total, 1), correct / max(total, 1), grad_norm, gd_update_norm


def train_gd_thaler(config: GDThalerConfig, verbose=True):
    set_seed(config.seed)

    train_loader, test_loader, _, _ = get_dataloaders(
        dataset=config.dataset,
        train_batch_size=config.full_grad_batch_size,
        test_batch_size=512,
        shuffle_train=False,
        train_subset_size=config.train_subset_size,
        test_subset_size=TEST_SUBSET_SIZE,
        seed=config.seed,
    )

    model = build_fcn4(width=config.width).to(device)
    n_params = count_parameters(model)
    directions = make_deterministic_directions(model, seed=config.direction_seed)
    generators = make_thaler_generators_for_model(model, config)

    history = []
    noise_rows = []
    start = time.time()

    stopped_by_nan = False
    stopped_by_early = False
    last_valid_state = copy.deepcopy(model.state_dict())

    # Initial evaluation: checkpoint can be initialized safely.
    init_test_loss, init_test_acc = evaluate(model, test_loader)
    best_state = copy.deepcopy(model.state_dict())
    best_test_acc = float(init_test_acc)
    best_test_loss = float(init_test_loss)
    best_step = 0
    bad_evals_after_target = 0

    history.append({
        "step": 0,
        "train_loss": np.nan,
        "train_acc": np.nan,
        "test_loss": init_test_loss,
        "test_acc": init_test_acc,
        "best_test_acc": best_test_acc,
        "best_step": best_step,
        "thaler_z_mean": np.nan,
        "thaler_z_max_abs": np.nan,
        "raw_thaler_mean": np.nan,
        "effective_noise_norm": 0.0,
        "grad_norm": np.nan,
        "gd_update_norm": np.nan,
        "noise_to_gd_update_ratio": 0.0,
        "elapsed_sec": 0.0,
    })

    pbar = tqdm(range(1, config.max_gd_steps + 1), desc=f"GD+Thaler {config.noise_mode}", leave=True)
    for step in pbar:
        train_loss, train_acc, grad_norm, gd_update_norm = full_gradient_step(model, train_loader, lr=config.lr)

        z_by_name, raw_by_name, x_by_name = next_noise_values(generators, model)
        noise_norm = add_thaler_noise(model, directions, z_by_name=z_by_name, noise_scale=config.noise_scale)
        ratio = noise_norm / max(gd_update_norm, 1e-12)

        # Check finite parameters.
        finite_params = True
        with torch.no_grad():
            for p in model.parameters():
                if not torch.isfinite(p).all():
                    finite_params = False
                    break

        if not finite_params or not np.isfinite(train_loss):
            print(f"NaN/Inf at step={step}; restoring last valid state.")
            model.load_state_dict(last_valid_state)
            stopped_by_nan = True
            break
        else:
            last_valid_state = copy.deepcopy(model.state_dict())

        z_vals = np.array(list(z_by_name.values()), dtype=np.float64) if z_by_name else np.array([0.0])
        raw_vals = np.array(list(raw_by_name.values()), dtype=np.float64) if raw_by_name else np.array([0.0])
        x_vals = np.array(list(x_by_name.values()), dtype=np.float64) if x_by_name else np.array([np.nan])

        noise_rows.append({
            "step": step,
            "thaler_z_mean": float(np.mean(z_vals)),
            "thaler_z_max_abs": float(np.max(np.abs(z_vals))),
            "raw_thaler_mean": float(np.mean(raw_vals)),
            "raw_thaler_max_abs": float(np.max(np.abs(raw_vals))),
            "x_state_mean": float(np.nanmean(x_vals)),
            "effective_noise_norm": noise_norm,
            "grad_norm": grad_norm,
            "gd_update_norm": gd_update_norm,
            "noise_to_gd_update_ratio": ratio,
        })

        do_eval = (step == 1 or step % config.log_every == 0 or step == config.max_gd_steps)
        if do_eval:
            test_loss, test_acc = evaluate(model, test_loader)

            improved = test_acc > best_test_acc
            if improved:
                best_test_acc = float(test_acc)
                best_test_loss = float(test_loss)
                best_step = int(step)
                best_state = copy.deepcopy(model.state_dict())
                bad_evals_after_target = 0
            else:
                if best_test_acc >= config.target_test_acc:
                    bad_evals_after_target += 1

            row = {
                "step": step,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "test_loss": test_loss,
                "test_acc": test_acc,
                "best_test_acc": best_test_acc,
                "best_step": best_step,
                "thaler_z_mean": float(np.mean(z_vals)),
                "thaler_z_max_abs": float(np.max(np.abs(z_vals))),
                "raw_thaler_mean": float(np.mean(raw_vals)),
                "effective_noise_norm": noise_norm,
                "grad_norm": grad_norm,
                "gd_update_norm": gd_update_norm,
                "noise_to_gd_update_ratio": ratio,
                "elapsed_sec": time.time() - start,
            }
            history.append(row)
            pbar.set_postfix({
                "train_acc": f"{train_acc:.3f}",
                "test_acc": f"{test_acc:.3f}",
                "best": f"{best_test_acc:.3f}",
                "ratio": f"{ratio:.2e}",
                "zmax": f"{np.max(np.abs(z_vals)):.2f}",
            })
            if verbose:
                print(row)

            if (
                config.use_best_checkpoint
                and best_test_acc >= config.target_test_acc
                and step >= config.min_steps_before_stop
                and bad_evals_after_target >= config.patience_evals
            ):
                print(
                    f"Early stopping at step={step}: best_test_acc={best_test_acc:.4f} "
                    f"at step={best_step}, no improvement for {bad_evals_after_target} evals."
                )
                stopped_by_early = True
                break

    # For pruning and alpha estimation use the best checkpoint, not the last state.
    if config.use_best_checkpoint and best_state is not None:
        model.load_state_dict(best_state)

    final_test_loss, final_test_acc = evaluate(model, test_loader)
    mean_alpha, alpha_df = estimate_model_alphas(model, skip_last=True)

    summary = asdict(config)
    summary.update({
        "beta": config.beta,
        "n_params": n_params,
        "train_size": len(train_loader.dataset),
        "test_size": len(test_loader.dataset),
        "final_test_loss": final_test_loss,
        "final_test_acc": final_test_acc,
        "best_test_loss": best_test_loss,
        "best_test_acc": best_test_acc,
        "best_step": best_step,
        "mean_alpha": mean_alpha,
        "train_time_sec": time.time() - start,
        "stopped_by_nan": stopped_by_nan,
        "stopped_by_early": stopped_by_early,
        "status": "nan" if stopped_by_nan else "ok",
        "used_checkpoint": "best" if config.use_best_checkpoint else "last",
    })

    # Store rough generator calibration diagnostics.
    if generators:
        first_gen = next(iter(generators.values()))
        summary.update({
            "noise_center": first_gen.center,
            "noise_scale_norm": first_gen.scale,
            "calib_raw_mean": first_gen.calib_mean,
            "calib_raw_std": first_gen.calib_std,
            "calib_raw_min": first_gen.calib_min,
            "calib_raw_max": first_gen.calib_max,
        })
    else:
        summary.update({
            "noise_center": np.nan,
            "noise_scale_norm": np.nan,
            "calib_raw_mean": np.nan,
            "calib_raw_std": np.nan,
            "calib_raw_min": np.nan,
            "calib_raw_max": np.nan,
        })

    return model, pd.DataFrame(history), pd.DataFrame(noise_rows), alpha_df, summary

In [ ]:
# 6. Запуск основного эксперимента
# Задаётся один основной режим параметров и сохраняются история обучения, шум, оценки хвостов и контрольная точка.

# Основной запуск, использованный в дипломной работе.
THALER_SWEEP_SPECS = [
    {
        "name": "thaler_medium",
        "lr": 0.03,
        "noise_scale": 0.0025,
        "s": 0.70,
        "target_alpha_noise": 1.40,
        "eps": 0.10,
        "inner_steps": 20,
        "x0": 0.323456789,
    },
]

if RUN_THALER_SWEEP:
    all_summaries = []

    for i, spec in enumerate(THALER_SWEEP_SPECS, start=1):
        cfg = GDThalerConfig(
            width=WIDTH,
            lr=spec["lr"],
            noise_scale=spec["noise_scale"],
            seed=SEED + i,
            direction_seed=2026 + i,
            use_thaler=True,
            noise_mode="layerwise",
            s=spec["s"],
            target_alpha_noise=spec["target_alpha_noise"],
            eps=spec["eps"],
            x0=spec["x0"],
            burn_in=1000,
            inner_steps=spec["inner_steps"],
            aggregate="sum_sqrt",
            clip_raw=1e6,
            calibrate_size=8000,
            train_subset_size=TRAIN_SUBSET_SIZE,
            full_grad_batch_size=FULL_GRAD_BATCH_SIZE,
            max_gd_steps=MAX_GD_STEPS,
            log_every=LOG_EVERY,
            target_test_acc=TARGET_TEST_ACC,
            patience_evals=PATIENCE_EVALS,
            min_steps_before_stop=MIN_STEPS_BEFORE_STOP,
            use_best_checkpoint=USE_BEST_CHECKPOINT,
        )

        print("\n" + "=" * 80)
        print(f"GD+Thaler experiment {i}/{len(THALER_SWEEP_SPECS)}: {spec['name']}")
        print(asdict(cfg) | {"beta": cfg.beta})

        # Separate noise diagnostics before training.
        diag, arr = diagnose_thaler_signal(asdict(cfg) | {"beta": cfg.beta}, n=5000)
        diag["name"] = spec["name"]
        pd.DataFrame([diag]).to_csv(os.path.join(RESULTS_DIR, f"noise_diag_{i:02d}_{spec['name']}.csv"), index=False)
        pd.DataFrame(arr, columns=["z", "raw", "x_state"]).to_csv(os.path.join(RESULTS_DIR, f"noise_sample_{i:02d}_{spec['name']}.csv"), index=False)
        print("Noise diagnostics:", diag)

        # Plot the first 1000 normalized noise values.
        plt.figure(figsize=(9, 3.5))
        plt.plot(arr[:1000, 0], linewidth=0.8)
        plt.xlabel("iteration")
        plt.ylabel("normalized z")
        plt.title(
            f"{spec['name']}: s={cfg.s}, beta={cfg.beta:.3f}, "
            f"target alpha={cfg.target_alpha_noise}, sigma={cfg.noise_scale}"
        )
        plt.grid(alpha=0.3)
        plt.tight_layout()
        noise_fig = os.path.join(FIGURES_DIR, f"thaler_noise_series_{i:02d}_{spec['name']}.png")
        plt.savefig(noise_fig, dpi=200)
        plt.show()

        model, hist_df, noise_df, alpha_df, summary = train_gd_thaler(cfg, verbose=True)
        summary["sweep_id"] = i
        summary["name"] = spec["name"]
        all_summaries.append(summary)

        hist_df.to_csv(os.path.join(RESULTS_DIR, f"gd_thaler_history_{i:02d}_{spec['name']}.csv"), index=False)
        noise_df.to_csv(os.path.join(RESULTS_DIR, f"gd_thaler_noise_used_{i:02d}_{spec['name']}.csv"), index=False)
        alpha_df.to_csv(os.path.join(RESULTS_DIR, f"gd_thaler_layer_alpha_{i:02d}_{spec['name']}.csv"), index=False)

        ckpt_path = os.path.join(CHECKPOINTS_DIR, f"gd_thaler_fcn4_mnist_{i:02d}_{spec['name']}.pt")
        torch.save({
            "model_state_dict": model.state_dict(),  # сохраняется лучшая модель, если USE_BEST_CHECKPOINT=True
            "summary": summary,
            "history": hist_df.to_dict(orient="records"),
            "noise_used": noise_df.to_dict(orient="records"),
            "layer_alpha": alpha_df.to_dict(orient="records"),
        }, ckpt_path)
        print("Saved checkpoint:", ckpt_path)

    summary_df = pd.DataFrame(all_summaries)
    summary_df = summary_df.sort_values(["sweep_id"])
    summary_path = os.path.join(RESULTS_DIR, "gd_thaler_sweep_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    cols = [
        "sweep_id", "name", "width", "lr", "noise_scale", "s", "target_alpha_noise", "beta",
        "eps", "inner_steps", "noise_mode", "best_step", "best_test_acc", "final_test_acc",
        "mean_alpha", "status", "stopped_by_early", "train_time_sec"
    ]
    display(summary_df[[c for c in cols if c in summary_df.columns]])
    print("Saved:", summary_path)
else:
    print("RUN_THALER_SWEEP=False: sweep skipped")

In [ ]:
# 7. Графики по результатам обучения
# Строятся графики хвостового показателя и качества модели.

summary_path = os.path.join(RESULTS_DIR, "gd_thaler_sweep_summary.csv")
if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    valid = df[(df["status"] == "ok") & df["mean_alpha"].notna()].copy()
    display(valid[["sweep_id", "s", "target_alpha_noise", "beta", "noise_scale", "final_test_acc", "mean_alpha"]])

    plt.figure(figsize=(7, 5))
    sc = plt.scatter(
        valid["target_alpha_noise"],
        valid["mean_alpha"],
        c=valid["noise_scale"],
        s=90,
        edgecolor="black",
        linewidth=0.5,
    )
    plt.xlabel(r"target noise tail index $\alpha_{noise}$")
    plt.ylabel(r"mean weight $\widehat{\alpha}$")
    plt.title("GD+Thaler: weight tail index vs deterministic-noise tail target")
    plt.grid(alpha=0.3)
    cbar = plt.colorbar(sc)
    cbar.set_label("noise_scale sigma")
    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, "gd_thaler_alpha_vs_noise_tail_target.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()
    print("Saved:", fig_path)

    plt.figure(figsize=(7, 5))
    sc = plt.scatter(
        valid["mean_alpha"],
        valid["final_test_acc"],
        c=valid["noise_scale"],
        s=90,
        edgecolor="black",
        linewidth=0.5,
    )
    plt.xlabel(r"mean weight $\widehat{\alpha}$")
    plt.ylabel("test accuracy")
    plt.title("GD+Thaler: test accuracy vs weight tail index")
    plt.grid(alpha=0.3)
    cbar = plt.colorbar(sc)
    cbar.set_label("noise_scale sigma")
    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, "gd_thaler_test_acc_vs_mean_alpha.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()
    print("Saved:", fig_path)
else:
    print("No summary file yet. Run the sweep cell first.")

In [ ]:
# 8. Методы разреживания
# Реализуются глобальное и послойное разреживание по модулю, разреживание по нейронам и по сингулярным числам.

def get_weight_parameter_names(model, skip_last=True):
    names = [name for name, p in model.named_parameters() if ("weight" in name and p.ndim >= 2)]
    if skip_last and len(names) > 1:
        names = names[:-1]
    return names


def load_gd_thaler_checkpoint_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    summary = ckpt["summary"]
    model = build_fcn4(width=int(summary.get("width", WIDTH))).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, summary, ckpt


def apply_global_magnitude_pruning(model, keep_ratio, skip_last=True):
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)
    with torch.no_grad():
        scores = torch.cat([p.detach().abs().flatten() for name, p in pruned.named_parameters() if name in names])
        total = scores.numel()
        k = int(math.ceil(float(keep_ratio) * total))
        if k >= total:
            return pruned
        if k <= 0:
            threshold = torch.inf
        else:
            threshold = torch.topk(scores, k, largest=True).values[-1]
        for name, p in pruned.named_parameters():
            if name in names:
                mask = (p.detach().abs() >= threshold).to(p.dtype)
                p.mul_(mask)
    return pruned


def apply_layerwise_magnitude_pruning(model, keep_ratio, skip_last=True):
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)
    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name not in names:
                continue
            scores = p.detach().abs().flatten()
            total = scores.numel()
            k = int(math.ceil(float(keep_ratio) * total))
            if k >= total:
                continue
            if k <= 0:
                p.zero_()
                continue
            threshold = torch.topk(scores, k, largest=True).values[-1]
            mask = (p.detach().abs() >= threshold).to(p.dtype)
            p.mul_(mask)
    return pruned


def apply_node_pruning(model, keep_ratio, skip_last=True):
    """
    Structured node/column pruning for Linear layers:
    remove columns with smallest l2 norm in each weight matrix.
    """
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)
    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name not in names or p.ndim != 2:
                continue
            col_norms = torch.linalg.norm(p.detach(), dim=0)
            total = col_norms.numel()
            k = int(math.ceil(float(keep_ratio) * total))
            if k >= total:
                continue
            if k <= 0:
                p.zero_()
                continue
            threshold = torch.topk(col_norms, k, largest=True).values[-1]
            mask_cols = (col_norms >= threshold).to(p.dtype)
            p.mul_(mask_cols.view(1, -1))
    return pruned


def apply_svd_pruning(model, keep_ratio, skip_last=True):
    """
    Low-rank/SVD pruning for each weight matrix:
    keep ceil(keep_ratio * rank) largest singular values.
    """
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)
    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name not in names or p.ndim != 2:
                continue
            W = p.detach()
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            except RuntimeError:
                U, S, Vh = torch.linalg.svd(W.cpu(), full_matrices=False)
                U, S, Vh = U.to(device), S.to(device), Vh.to(device)
            r = S.numel()
            k = int(math.ceil(float(keep_ratio) * r))
            if k >= r:
                continue
            if k <= 0:
                p.zero_()
                continue
            S_pruned = S.clone()
            S_pruned[k:] = 0.0
            W_new = (U * S_pruned.unsqueeze(0)) @ Vh
            p.copy_(W_new)
    return pruned


def run_pruning_for_checkpoint(ckpt_path, keep_ratios=PRUNING_KEEP_RATIOS, skip_last=True):
    base_model, summary, ckpt = load_gd_thaler_checkpoint_model(ckpt_path)
    base_loss, base_acc = evaluate(base_model, test_loader)

    methods = {
        "global_magnitude": apply_global_magnitude_pruning,
        "layerwise_magnitude": apply_layerwise_magnitude_pruning,
        "node": apply_node_pruning,
    }
    if RUN_SVD_PRUNING:
        methods["svd"] = apply_svd_pruning

    rows = []
    for method_name, method_fn in methods.items():
        for keep_ratio in keep_ratios:
            pruned_model = method_fn(base_model, keep_ratio=keep_ratio, skip_last=skip_last)
            loss, acc = evaluate(pruned_model, test_loader)
            rows.append({
                "checkpoint": os.path.basename(ckpt_path),
                "sweep_id": summary.get("sweep_id"),
                "method": method_name,
                "keep_ratio": keep_ratio,
                "pruning_ratio": 1.0 - keep_ratio,
                "base_test_acc": base_acc,
                "test_acc": acc,
                "relative_test_acc": acc / base_acc if base_acc > 0 else np.nan,
                "mean_alpha": summary.get("mean_alpha"),
                "s": summary.get("s"),
                "target_alpha_noise": summary.get("target_alpha_noise"),
                "beta": summary.get("beta"),
                "noise_scale": summary.get("noise_scale"),
                "width": summary.get("width"),
                "lr": summary.get("lr"),
            })
            print(method_name, "keep", keep_ratio, "acc", acc, "rel", rows[-1]["relative_test_acc"])
    return pd.DataFrame(rows)


if RUN_PRUNING:
    ckpt_paths = sorted([
        os.path.join(CHECKPOINTS_DIR, f)
        for f in os.listdir(CHECKPOINTS_DIR)
        if f.endswith(".pt") and f.startswith("gd_thaler")
    ])
    print("Found checkpoints:", len(ckpt_paths))

    all_pruning = []
    for ckpt_path in ckpt_paths:
        print("\nPruning:", ckpt_path)
        pr_df = run_pruning_for_checkpoint(ckpt_path)
        out_path = os.path.join(RESULTS_DIR, os.path.basename(ckpt_path).replace(".pt", "_pruning.csv"))
        pr_df.to_csv(out_path, index=False)
        all_pruning.append(pr_df)

    if all_pruning:
        pruning_df = pd.concat(all_pruning, ignore_index=True)
        pruning_path = os.path.join(RESULTS_DIR, "gd_thaler_pruning_all.csv")
        pruning_df.to_csv(pruning_path, index=False)
        display(pruning_df.head())
        print("Saved:", pruning_path)
else:
    print("RUN_PRUNING=False: pruning skipped")

In [ ]:
# 9. Графики разреживания
# Строятся зависимости относительной точности от доли удалённых параметров.

pruning_path = os.path.join(RESULTS_DIR, "gd_thaler_pruning_all.csv")
if os.path.exists(pruning_path):
    pruning_df = pd.read_csv(pruning_path)
    display(pruning_df[[
        "sweep_id", "method", "pruning_ratio", "relative_test_acc", "test_acc",
        "mean_alpha", "s", "target_alpha_noise", "noise_scale"
    ]].head(20))

    for method_name, g in pruning_df.groupby("method"):
        plt.figure(figsize=(7, 5))
        for sweep_id, gg in g.groupby("sweep_id"):
            gg = gg.sort_values("pruning_ratio")
            label = (
                f"id={sweep_id}, alpha_w={gg['mean_alpha'].iloc[0]:.3f}, "
                f"a_noise={gg['target_alpha_noise'].iloc[0]:.2f}"
            )
            plt.plot(gg["pruning_ratio"], gg["relative_test_acc"], marker="o", linewidth=1.3, label=label)
        plt.xlabel("pruning ratio (1 - keep_ratio)")
        plt.ylabel("relative test accuracy")
        plt.title(f"GD+Thaler pruning: {method_name}")
        plt.ylim(0, 1.05)
        plt.grid(alpha=0.3)
        plt.legend(fontsize=8)
        plt.tight_layout()
        fig_path = os.path.join(FIGURES_DIR, f"gd_thaler_pruning_{method_name}.png")
        plt.savefig(fig_path, dpi=200)
        plt.show()
        print("Saved:", fig_path)

    # Точечный график: относительная точность после разреживания и средний хвостовой показатель.
    for method_name, g in pruning_df.groupby("method"):
        plt.figure(figsize=(7, 5))
        sc = plt.scatter(
            g["pruning_ratio"],
            g["relative_test_acc"],
            c=g["mean_alpha"],
            s=65,
            edgecolor="black",
            linewidth=0.4,
        )
        plt.xlabel("pruning ratio (1 - keep_ratio)")
        plt.ylabel("relative test accuracy")
        plt.title(f"GD+Thaler: relative accuracy vs pruning ({method_name})")
        plt.ylim(0, 1.05)
        plt.grid(alpha=0.3)
        cbar = plt.colorbar(sc)
        cbar.set_label(r"mean weight $\widehat{\alpha}$")
        plt.tight_layout()
        fig_path = os.path.join(FIGURES_DIR, f"gd_thaler_pruning_scatter_{method_name}.png")
        plt.savefig(fig_path, dpi=200)
        plt.show()
        print("Saved:", fig_path)
else:
    print("No pruning file yet. Run the pruning cell first.")

In [ ]:
# 10. Экспорт результатов
# Все таблицы, графики и контрольные точки упаковываются в zip-архив.

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"vkr_gd_thaler_results_{timestamp}.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in [RESULTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for fn in files:
                path = os.path.join(root, fn)
                zf.write(path, arcname=path)

print("Created:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Если это не Colab, файл лежит в текущей папке:", zip_name)